# [1.6] Local Frontier ML Infrastructure - Solutions

Reference validation notebook for the local frontier-infrastructure harness.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

chapter = "chapter1_transformer_interp"
section = "part6_frontier_ml_infrastructure"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_frontier_ml_infrastructure.tests as tests
from chapter1_transformer_interp.exercises.part6_frontier_ml_infrastructure import solutions

In [ ]:
tests.test_memory_budget_fits_local_tier(solutions.estimate_gemma_1b_smoke_budget)
tests.test_compare_logits_detects_match(solutions.compare_logits)
tests.test_compare_logits_rejects_shape_mismatch(solutions.compare_logits)
tests.test_hf_parity_smoke_test_passes(solutions.hf_parity_smoke_test)
tests.test_deterministic_generation_equal_detects_mismatch(
    solutions.deterministic_generation_equal,
)
with tempfile.TemporaryDirectory() as tmpdir:
    tests.test_disk_activation_store_roundtrip(Path(tmpdir))
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["hf_parity_passed"]
assert contract["generation_parity_passed"]
assert contract["budget"]["total_gb"] < 4.0
assert contract["environment"]["torch"].endswith("+cu132")
contract

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    activation_summary = solutions.activation_store_smoke_test(Path(tmpdir))
assert activation_summary["num_records"] == 2
assert activation_summary["names"] == ["resid_pre", "mlp_out"]
activation_summary

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["python_major_minor"] == "3.14"
assert gpu["torch_version"] == "2.12.1+cu132"
assert gpu["torchvision_version"] == "0.27.1+cu132"
assert gpu["cuda_version"] == "13.2"
assert gpu["bf16_supported"]
assert gpu["estimated_total_gb"] < 4.0
assert gpu["fits_budget"]
assert gpu["gpu_tensor_test_passed"]
assert gpu["gpu_matmul_shape"] == [1024, 1024]
assert gpu["gpu_matmul_dtype"] == "torch.bfloat16"
assert gpu["gpu_matmul_finite"]
assert gpu["gpu_matmul_mean_abs"] > 1.0
assert gpu["gpu_matmul_diag_mean"] > 100.0
assert gpu["uv_pip_check_passed"]
assert gpu["uv_pip_check_returncode"] == 0
assert gpu["peak_vram_gb"] < 1.0
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "gpu_name",
    "python_version",
    "torch_version",
    "torchvision_version",
    "cuda_version",
    "gpu_matmul_dtype",
    "gpu_matmul_diag_mean",
    "peak_vram_gb",
]}